# Tarea 2 · Parte 2 – Evaluación

**Muestra fija** (definida antes de medir, en `datos/muestra_evaluacion.csv`): 10 documentos (3 impresos, 3 fotos, 4 manuscritos) y 10 clips (4 de OpenSLR 73 y los 6 más cortos de Commons), priorizando archivos con borrador de referencia y textos cortos.

**Referencias:** transcripción manual de los 10 documentos mirando cada imagen (`datos/raw/referencias/`, seudonimizadas con la misma función de la Parte 1). Las referencias manuales de los 10 clips quedan **pendientes**: por eso en esta versión solo se reporta el CER.

In [1]:
import json, re, time
from pathlib import Path
import numpy as np, pandas as pd, plotly.express as px, plotly.io as pio
import jiwer
import voz
from voz import SR, cargar_audio, normalizar, Transcriptor
pio.renderers.default = "vscode+png"

DATOS = Path("datos")
docs = pd.read_csv(DATOS / "documentos.csv").fillna({"texto_referencia": ""})
auds = pd.read_csv(DATOS / "transcripciones.csv").fillna({"texto_referencia": ""})

REF = DATOS / "raw" / "referencias"
def con_ref(df):
    "True si ya hay referencia o borrador (⚡ manuscritos propios y clips de OpenSLR)."
    BORRADOR = DATOS / "raw" / "referencias_borrador"
    return df["id"].map(lambda i: (REF / f"{i}.txt").exists() or (BORRADOR / f"{i}.txt").exists())

# ⚡ Criterio explícito (escríbelo en Markdown): primero los archivos que ya tienen referencia o borrador,
# luego los de texto más corto; en audio, 4 clips de OpenSLR (estudio) + los 6 más cortos de Commons (calle).
POR_TIPO = {"impreso": 3, "foto": 3, "manuscrito": 4}
docs["_ref"], docs["_largo"] = con_ref(docs), docs.texto_ocr.fillna("").str.len()
muestra_docs = pd.concat([docs[docs.tipo == t].sort_values(["_ref", "_largo"], ascending=[False, True]).head(n)
                          for t, n in POR_TIPO.items()])
docs = docs.drop(columns=["_ref", "_largo"]); muestra_docs = muestra_docs.drop(columns=["_ref", "_largo"])
muestra_auds = pd.concat([auds[con_ref(auds)].sample(4, random_state=42),
                          auds[~con_ref(auds)].nsmallest(6, "duracion_seg")])
RUTA_MUESTRA = DATOS / "muestra_evaluacion.csv"
if RUTA_MUESTRA.exists():          # ♻️ la muestra se fija una sola vez: al re-ejecutar se lee, no se vuelve a elegir
    muestra = pd.read_csv(RUTA_MUESTRA)
    muestra_docs = docs[docs.id.isin(muestra.id)]
    muestra_auds = auds[auds.id.isin(muestra.id)]
else:
    pd.concat([muestra_docs[["id", "tipo"]].assign(modalidad="documento"),
               muestra_auds[["id"]].assign(modalidad="audio")]).to_csv(RUTA_MUESTRA, index=False)
print(len(muestra_docs), "documentos y", len(muestra_auds), "clips en la muestra")

10 documentos y 10 clips en la muestra


In [2]:
from voz import seudonimizar
RUTA_REEMPLAZOS = DATOS / "raw" / "reemplazos.json"
REEMPLAZOS = json.loads(RUTA_REEMPLAZOS.read_text(encoding="utf-8")) if RUTA_REEMPLAZOS.exists() else {}
def aplicar(texto):
    "Seudonimiza y devuelve (texto nuevo, número de reemplazos)."
    texto = str(texto)
    nuevo = seudonimizar(texto, REEMPLAZOS) if REEMPLAZOS else texto
    nuevo = re.sub(r"\b\d{8}\b", "[DNI]", nuevo)
    n = sum(nuevo.count(s) - texto.count(s) for s in set(REEMPLAZOS.values()) | {"[DNI]"})
    return nuevo, n

def seud(t):
    return aplicar(t)[0]

REF = DATOS / "raw" / "referencias"
def referencia(i, publicada):
    "Lee tu .txt privado si existe; si no (♻️ otra persona re-ejecutando), conserva la referencia ya publicada."
    ruta = REF / f"{i}.txt"
    return seud(ruta.read_text(encoding="utf-8").strip()) if ruta.exists() else publicada

docs["texto_referencia"] = [referencia(i, r) for i, r in zip(docs.id, docs.texto_referencia)]
auds["texto_referencia"] = [referencia(i, r) for i, r in zip(auds.id, auds.texto_referencia)]
docs.to_csv(DATOS / "documentos.csv", index=False)
auds.to_csv(DATOS / "transcripciones.csv", index=False)
print((docs.texto_referencia != "").sum(), "documentos y", (auds.texto_referencia != "").sum(), "clips con referencia")

10 documentos y 0 clips con referencia


In [3]:
def sin_marcas(t):
    return re.sub(r"\[HABLANTE_\d+\]", " ", str(t or ""))

def cer(ref, hip):
    return jiwer.cer(normalizar(ref), normalizar(hip) or "∅")

def wer(ref, hip):
    return jiwer.wer(normalizar(sin_marcas(ref)), normalizar(sin_marcas(hip)) or "∅")

resultados = []          # una fila por (archivo, modelo)

In [4]:
md = docs[docs.id.isin(muestra_docs.id) & (docs.texto_referencia != "")].copy()
for f in md.itertuples():
    resultados.append({"id": f.id, "modalidad": "documento", "tipo": f.tipo, "modelo": "PaddleOCR PP-OCRv6 (base)",
                       "texto": f.texto_ocr, "error": cer(f.texto_referencia, f.texto_ocr)})

In [5]:
from faster_whisper import WhisperModel

ma = auds[auds.id.isin(muestra_auds.id) & (auds.texto_referencia != "")].copy()

def medir_faster_whisper(pesos, etiqueta):
    "Transcribe los 10 clips con un modelo faster-whisper en CPU y guarda WER, segundos y RTF."
    modelo = WhisperModel(pesos, device="cpu", compute_type="int8")
    modelo.transcribe(cargar_audio(ma.archivo_audio.iloc[0])[:5 * SR], language="es")   # calentamiento
    for f in ma.itertuples():
        x = cargar_audio(f.archivo_audio)
        t0 = time.perf_counter()
        segs, _ = modelo.transcribe(x, language="es", beam_size=5, vad_filter=True)
        texto = seud(" ".join(s.text.strip() for s in segs))     # transcribe() es perezoso: aquí se ejecuta
        seg = time.perf_counter() - t0
        resultados.append({"id": f.id, "modalidad": "audio", "modelo": etiqueta, "texto": texto,
                           "error": wer(f.texto_referencia, texto),
                           "segundos": round(seg, 1), "rtf": round(seg / f.duracion_seg, 3)})

if len(ma):
    medir_faster_whisper("small", "faster-whisper small (CPU, int8, VAD) · base")
else:
    print("Audio: aún no hay referencias manuales de los clips; el WER queda pendiente.")

Audio: aún no hay referencias manuales de los clips; el WER queda pendiente.


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## OCR alternativo: EasyOCR

El enunciado permite «Tesseract u otro modelo de lectura de texto debidamente justificado». Se usa **EasyOCR**: un OCR abierto con otra arquitectura que PaddleOCR (detector CRAFT + reconocedor CRNN, sobre PyTorch), que se instala solo con uv (sin programas del sistema) y corre en CPU. Se mide en los **mismos 10 documentos**, sin ajustar parámetros.

In [6]:
import easyocr

lector = easyocr.Reader(["es"], gpu=False)   # CPU; se crea una sola vez

for f in md.itertuples():
    t0 = time.perf_counter()
    texto = seud(" ".join(" ".join(lector.readtext(str(f.archivo), detail=0, paragraph=True)).split()))
    resultados.append({"id": f.id, "modalidad": "documento", "tipo": f.tipo, "modelo": "EasyOCR (es, CPU)",
                       "texto": texto, "error": cer(f.texto_referencia, texto),
                       "segundos": round(time.perf_counter() - t0, 2)})
print(len(resultados), "mediciones")

Using CPU. Note: This module is much faster with a GPU.


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/josselinyauri/Documents/GitHub/python-actividades/tarea_2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


20 mediciones


## Comparativa

In [7]:
res = pd.DataFrame(resultados)
res.to_csv(DATOS / "evaluacion_por_archivo.csv", index=False)
tabla = (res.groupby(["modalidad", "modelo"])
            .agg(error_promedio=("error", "mean"), segundos=("segundos", "mean"))
            .round(3).reset_index())
tabla["metrica"] = np.where(tabla.modalidad == "documento", "CER", "WER")
tabla["rtf"] = np.nan
tabla["elegido"] = tabla.error_promedio == tabla.groupby("modalidad").error_promedio.transform("min")
tabla.assign(valor=tabla.error_promedio)[["modalidad", "modelo", "metrica", "valor", "rtf", "elegido"]].to_csv(DATOS / "metricas.csv", index=False)
display(tabla)
display(res[res.modalidad == "documento"].pivot_table(index="tipo", columns="modelo", values="error", aggfunc="mean").round(3))

,modalidad,modelo,error_promedio,segundos,metrica,rtf,elegido
0,documento,"EasyOCR (es, CPU)",0.878,4.202,CER,NaN,False
1,documento,PaddleOCR PP-OCRv6 (base),0.488,NaN,CER,NaN,True


modelo,"EasyOCR (es, CPU)",PaddleOCR PP-OCRv6 (base)
tipo,,
foto,0.348,0.136
impreso,1.955,0.971
manuscrito,0.467,0.391


In [8]:
m = pd.read_csv(DATOS / "metricas.csv").query("elegido")
cer_txt = f"{m[m.metrica == 'CER'].valor.iloc[0]:.2f}" if (m.metrica == "CER").any() else "pendiente"
wer_txt = f"{m[m.metrica == 'WER'].valor.iloc[0]:.2f}" if (m.metrica == "WER").any() else "pendiente"
print(f"WER: {wer_txt} | CER: {cer_txt} | Acuerdo LLM-Humano: pendiente")

WER: pendiente | CER: 0.49 | Acuerdo LLM-Humano: pendiente


## Pendiente

- Referencias manuales de los 10 clips y WER de faster-whisper `small` frente a Qwen3-ASR.
- Extracción estructurada con Gemini (`extraccion_llm.csv`) y auditoría humana de 20 extracciones.
- Casos difíciles y decisión final por modalidad.